<a href="https://colab.research.google.com/github/Dana-El/CCI-HW5/blob/main/Lab2_Tools_Agent_Loop_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session5/student/Lab2_Tools_Agent_Loop_Student.ipynb)

# Lab 2: Tools & the Agent Loop
## CCI Session 5

**Duration:** 15 minutes

### Clinical Scenario
> Build a clinical assistant with tools that look up patient data, check vitals reference ranges, and flag drug interactions. Compare with the manual tool calling you built in Session 3 Lab 3.

### Objective
- Define clinical tools with `@tool` decorator
- Build an agent with multiple tools
- Observe the agent loop in action
- Add error handling and guardrails

---
## Setup

In [ ]:
!pip install -q langchain langchain-openai langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.1 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup complete.")

Setup complete.


---
## Cell 1 — Define Clinical Tools with @tool

In Session 3, you defined tools as JSON schemas and wrote manual dispatch logic:
```python
tools = [{
    "type": "function",
    "function": {
        "name": "lookup_patient",
        "parameters": { ... }  # Manual JSON schema
    }
}]
```

With LangChain's `@tool` decorator, the schema is generated automatically from the function signature and docstring.

In [ ]:
from langchain_core.tools import tool

# --- Tool 1: Patient Lookup ---
@tool
def lookup_patient(mrn: str) -> dict:
    """Look up a patient record by their Medical Record Number (MRN).
    Returns patient demographics and current diagnosis."""
    patients = {
        "MRN001": {
            "name": "Ahmad Al-Rashid",
            "age": 58,
            "diagnosis": "Non-Small Cell Lung Cancer (NSCLC), Stage IIIA",
            "current_medications": ["cisplatin", "pemetrexed", "ondansetron"],
            "allergies": ["penicillin"]
        },
        "MRN002": {
            "name": "Fatima Hassan",
            "age": 45,
            "diagnosis": "Chronic Myeloid Leukemia (CML), Chronic Phase",
            "current_medications": ["imatinib", "allopurinol"],
            "allergies": []
        },
        "MRN003": {
            "name": "Yousef Bakri",
            "age": 67,
            "diagnosis": "Diffuse Large B-Cell Lymphoma (DLBCL), Stage II",
            "current_medications": ["rituximab", "cyclophosphamide", "doxorubicin", "vincristine", "prednisone"],
            "allergies": ["sulfa"]
        }
    }
    patient = patients.get(mrn.upper())
    if patient:
        return patient
    return {"error": f"Patient with MRN {mrn} not found"}

# --- Tool 2: Vitals Check ---
@tool
def check_vitals(mrn: str) -> dict:
    """Check the vital signs for a given patient by their MRN.
    Returns temperature, heart rate, blood pressure, WBC, and platelets."""
    vitals = {
        "MRN001": {"temp": 38.5, "heart_rate": 92, "bp": "130/85", "wbc": 3.2, "platelets": 145},
        "MRN002": {"temp": 36.8, "heart_rate": 78, "bp": "120/80", "wbc": 8.5, "platelets": 210},
        "MRN003": {"temp": 37.1, "heart_rate": 85, "bp": "140/90", "wbc": 2.8, "platelets": 95}
    }
    patient_vitals = vitals.get(mrn.upper())
    if patient_vitals:
        return patient_vitals
    return {"error": f"Vitals for MRN {mrn} not found"}

# --- Tool 3: Drug Interaction Check ---
@tool
def check_drug_interaction(drug1: str, drug2: str) -> dict:
    """Check for drug interactions between two medications.
    Returns the severity of the interaction and a description."""
    interactions = {
        ("cisplatin", "pemetrexed"): {"severity": "moderate", "desc": "Monitor renal function closely"},
        ("imatinib", "allopurinol"): {"severity": "low", "desc": "No significant interaction"},
        ("rituximab", "cyclophosphamide"): {"severity": "moderate", "desc": "Increased immunosuppression risk"},
        ("methotrexate", "ibuprofen"): {"severity": "high", "desc": "Increased methotrexate toxicity risk"}
    }

    interaction = interactions.get((drug1.lower(), drug2.lower())) or interactions.get((drug2.lower(), drug1.lower()))
    if interaction:
        return interaction
    return {"severity": "unknown", "desc": "No interaction data found for these drugs."}

# Verify tools are properly defined
tools = [lookup_patient, check_vitals, check_drug_interaction]
for t in tools:
    print(f"Tool: {t.name}")
    print(f"  Description: {t.description}")
    print(f"  Schema: {t.args_schema.model_json_schema()}")
    print()


Tool: lookup_patient
  Description: Look up a patient record by their Medical Record Number (MRN).
    Returns patient demographics and current diagnosis.
  Schema: {'description': 'Look up a patient record by their Medical Record Number (MRN).\nReturns patient demographics and current diagnosis.', 'properties': {'mrn': {'title': 'Mrn', 'type': 'string'}}, 'required': ['mrn'], 'title': 'lookup_patient', 'type': 'object'}

Tool: check_vitals
  Description: Check the vital signs for a given patient by their MRN.
    Returns temperature, heart rate, blood pressure, WBC, and platelets.
  Schema: {'description': 'Check the vital signs for a given patient by their MRN.\nReturns temperature, heart rate, blood pressure, WBC, and platelets.', 'properties': {'mrn': {'title': 'Mrn', 'type': 'string'}}, 'required': ['mrn'], 'title': 'check_vitals', 'type': 'object'}

Tool: check_drug_interaction
  Description: Check for drug interactions between two medications.
    Returns the severity of the int

---
## Cell 2 — Create the Agent

Now let's create an agent that can use all three tools. We'll use `create_react_agent` which implements the ReAct (Reason + Act) pattern.

In [ ]:
from langgraph.prebuilt import create_react_agent

# System prompt to set clinical context
system_prompt = """You are a clinical decision support assistant at King Hussein Cancer Center (KHCC).
You have access to patient records, vitals, and drug interaction databases.
Always verify patient identity before providing clinical information.
Flag any concerning vitals or dangerous drug interactions prominently."""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

print("Agent created with", len(tools), "tools")


Agent created with 3 tools


/tmp/ipykernel_1272/1099649628.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


---
## Cell 3 — Invoke and Observe the Agent Loop

Watch how the agent reasons about which tools to call and in what order.

In [ ]:
# Ask a question that requires multiple tool calls
query = "Look up patient MRN001 and check their vitals. Are any vitals concerning?"

result = agent.invoke({"messages": [{"role": "user", "content": query}]})

# Print the full agent loop
print("=" * 60)
print("AGENT LOOP TRACE")
print("=" * 60)

for i, msg in enumerate(result["messages"]):
    role = msg.__class__.__name__
    print(f"\n--- Step {i}: [{role}] ---")

    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  TOOL CALL: {tc['name']}({tc['args']})")

    if msg.content:
        content = msg.content if len(msg.content) < 500 else msg.content[:500] + "..."
        print(f"  {content}")


AGENT LOOP TRACE

--- Step 0: [HumanMessage] ---
  Look up patient MRN001 and check their vitals. Are any vitals concerning?

--- Step 1: [AIMessage] ---
  TOOL CALL: lookup_patient({'mrn': 'MRN001'})
  TOOL CALL: check_vitals({'mrn': 'MRN001'})

--- Step 2: [ToolMessage] ---
  {"name": "Ahmad Al-Rashid", "age": 58, "diagnosis": "Non-Small Cell Lung Cancer (NSCLC), Stage IIIA", "current_medications": ["cisplatin", "pemetrexed", "ondansetron"], "allergies": ["penicillin"]}

--- Step 3: [ToolMessage] ---
  {"temp": 38.5, "heart_rate": 92, "bp": "130/85", "wbc": 3.2, "platelets": 145}

--- Step 4: [AIMessage] ---
  Patient Information:
- **Name:** Ahmad Al-Rashid
- **Age:** 58
- **Diagnosis:** Non-Small Cell Lung Cancer (NSCLC), Stage IIIA
- **Current Medications:** Cisplatin, Pemetrexed, Ondansetron
- **Allergies:** Penicillin

Vital Signs:
- **Temperature:** 38.5°C (elevated)
- **Heart Rate:** 92 bpm (normal)
- **Blood Pressure:** 130/85 mmHg (normal)
- **WBC:** 3.2 (low)
- **Platelets:

---
## Cell 4 — Multi-Turn Conversation

The agent maintains context across turns. Let's ask follow-up questions.

In [ ]:
# Use the messages from the previous result and add a follow-up question
follow_up_messages = result["messages"] + [
    {"role": "user", "content": "Check if there are any interactions between their current medications cisplatin and pemetrexed."}
]

result2 = agent.invoke({"messages": follow_up_messages})

# Print only the new messages (after the previous conversation)
new_messages = result2["messages"][len(result["messages"]):]
print("=" * 60)
print("FOLLOW-UP RESPONSE")
print("=" * 60)

for msg in new_messages:
    role = msg.__class__.__name__
    print(f"\n[{role}]")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  TOOL CALL: {tc['name']}({tc['args']})")
    if msg.content:
        print(f"  {msg.content[:500]}")


FOLLOW-UP RESPONSE

[HumanMessage]
  Check if there are any interactions between their current medications cisplatin and pemetrexed.

[AIMessage]
  TOOL CALL: check_drug_interaction({'drug1': 'cisplatin', 'drug2': 'pemetrexed'})

[ToolMessage]
  {"severity": "moderate", "desc": "Monitor renal function closely"}

[AIMessage]
  There is a **moderate interaction** between **cisplatin** and **pemetrexed**. 

### Interaction Details:
- **Description:** Monitor renal function closely.

It is important to ensure that renal function is regularly assessed while the patient is on these medications to prevent potential complications.


---
## Cell 5 — Iteration Limits and Error Handling

In production, you want to prevent infinite loops. Let's add safety guardrails.

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Look up MRN001, check vitals, and check all drug interactions between their medications."}]},
        config={"recursion_limit": 10}
    )
    # Print final answer
    final_msg = result["messages"][-1]
    print("Final answer:")
    print(final_msg.content[:500])
    print(f"\nTotal messages in loop: {len(result['messages'])}")

except GraphRecursionError:
    print("Agent hit the recursion limit! The task may be too complex.")
    print("Consider breaking it into smaller queries.")


Final answer:
### Patient Information
- **Name:** Ahmad Al-Rashid
- **Age:** 58
- **Diagnosis:** Non-Small Cell Lung Cancer (NSCLC), Stage IIIA
- **Current Medications:** 
  - Cisplatin
  - Pemetrexed
  - Ondansetron
- **Allergies:** Penicillin

### Vital Signs
- **Temperature:** 38.5°C (Elevated)
- **Heart Rate:** 92 bpm
- **Blood Pressure:** 130/85 mmHg
- **WBC:** 3.2 (Low)
- **Platelets:** 145 (Normal)

### Drug Interactions
1. **Cisplatin and Pemetrexed**
   - **Severity:** Moderate
   - **Description:** 

Total messages in loop: 9


---
## Cell 6 — Side-by-Side Comparison: Session 3 vs LangChain

Let's see how much simpler the LangChain approach is compared to Session 3.

In [ ]:
# This cell is for READING — don't run it (it uses the raw OpenAI client)
# It shows the Session 3 approach side-by-side with the LangChain approach

comparison = """
╔══════════════════════════════════════╦══════════════════════════════════════╗
║  SESSION 3: Raw OpenAI API          ║  SESSION 5: LangChain + LangGraph   ║
╠══════════════════════════════════════╬══════════════════════════════════════╣
║                                      ║                                      ║
║  # 1. Define tool schema manually    ║  # 1. Just use @tool decorator       ║
║  tools = [{                          ║  @tool                               ║
║    "type": "function",               ║  def lookup_patient(mrn: str) -> d:  ║
║    "function": {                     ║      \"\"\"Docstring = schema.\"\"\"   ║
║      "name": "lookup_patient",       ║      return patients[mrn]            ║
║      "parameters": {                 ║                                      ║
║        "type": "object",             ║                                      ║
║        "properties": { ... }         ║                                      ║
║      }                               ║                                      ║
║    }                                 ║                                      ║
║  }]                                  ║                                      ║
║                                      ║                                      ║
║  # 2. Manual tool dispatch           ║  # 2. Agent handles dispatch         ║
║  while True:                         ║  agent = create_react_agent(         ║
║    resp = client.chat.completions.   ║      model=llm,                      ║
║      create(...)                     ║      tools=[lookup_patient]          ║
║    if resp...tool_calls:             ║  )                                   ║
║      for tc in tool_calls:           ║                                      ║
║        name = tc.function.name       ║  # 3. Just invoke                    ║
║        args = json.loads(...)        ║  result = agent.invoke(              ║
║        if name == "lookup":          ║      {"messages": [query]}           ║
║          result = lookup(...)        ║  )                                   ║
║        messages.append(...)          ║                                      ║
║    else:                             ║                                      ║
║      break                           ║                                      ║
║                                      ║                                      ║
║  ~40 lines of code                   ║  ~10 lines of code                   ║
╚══════════════════════════════════════╩══════════════════════════════════════╝
"""
print(comparison)


╔══════════════════════════════════════╦══════════════════════════════════════╗
║  SESSION 3: Raw OpenAI API          ║  SESSION 5: LangChain + LangGraph   ║
╠══════════════════════════════════════╬══════════════════════════════════════╣
║                                      ║                                      ║
║  # 1. Define tool schema manually    ║  # 1. Just use @tool decorator       ║
║  tools = [{                          ║  @tool                               ║
║    "type": "function",               ║  def lookup_patient(mrn: str) -> d:  ║
║    "function": {                     ║      """Docstring = schema."""   ║
║      "name": "lookup_patient",       ║      return patients[mrn]            ║
║      "parameters": {                 ║                                      ║
║        "type": "object",             ║                                      ║
║        "properties": { ... }         ║                                      ║
║      }                               ║     

---
## Stretch Challenge

1. Add a 4th tool: `get_lab_results(mrn: str) -> dict` with mock CBC/chemistry data
2. Ask the agent: "Give me a complete clinical summary for MRN003 including labs, vitals, and any drug interactions"
3. Count how many tool calls the agent makes

### KHCC Connection
This pattern directly maps to a real clinical decision support system at KHCC:
- `lookup_patient` could connect to the Hospital Information System (HIS)
- `check_vitals` could pull from bedside monitors or nursing documentation
- `check_drug_interaction` could connect to a drug database like Lexicomp
- The agent loop automates the workflow a pharmacist does manually today